# Ditto Training — Complete RunPod Pipeline

**Ditto: Motion-Space Diffusion for Controllable Realtime Talking Head Synthesis**  
Full end-to-end notebook: environment setup → checkpoints → data preparation → training.

---
## Pipeline Overview

| Stage | Section | Description |
|-------|---------|-------------|
| 1 | [Setup](#1-environment-setup) | Clone repo, install Miniconda, create conda env, install all packages |
| 2 | [Checkpoints](#2-checkpoints-preparation) | Download model weights from HuggingFace |
| 3 | [Data Prep](#3-data-preparation) | Build `data_info.json`, run feature extraction pipeline |
| 4 | [Training](#4-model-training) | Configure Accelerate and launch MotionDiT training |

> **⚠️ Important:** Sections 1–2 run under the **default kernel**.  
> After Section 1 you must **register the `ditto_train` kernel, refresh the page, and switch kernels** before running Sections 2–4.

In [ ]:
!pip install huggingface_hub

In [ ]:
from huggingface_hub import snapshot_download
import os

repo_id = "Darknsu/HDTF_MEAD"
download_dir = "/workspace/"

# Create directory if it doesn't exist
os.makedirs(download_dir, exist_ok=True)

print("📥 Downloading HDTF dataset...")

snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    local_dir=download_dir,
    local_dir_use_symlinks=False,
)

print("✅ File downloaded to:", download_dir)

In [ ]:
!tar -xzvf /workspace/HDTF_MEAD.tar.gz -C /workspace/

In [ ]:
!mv /workspace/HDTF/* /workspace/ditto-train-lmdm-phase-2-v1/example/trainset_example/

In [ ]:
!rm -r /workspace/ditto-train-lmdm-phase-2-v1/example/trainset_example/fps25_mp4

In [ ]:
!cp -a /workspace/ditto-train-lmdm-phase-2-v1/example/trainset_example/video/. /workspace/ditto-train-lmdm-phase-2-v1/example/trainset_example/fps25_mp4/

---
# 1  Environment Setup

Run all cells in this section top-to-bottom on the **default (system) kernel**.

## 1.1  Clone Repository

In [ ]:
# !git clone https://github.com/B-I-T-W-I-S-E-M-I-N-D-S/ditto-train-fixed-for-HDTF.git /workspace/ditto-train-fixed-for-HDTF
# print("Repository cloned to /workspace/ditto-train-fixed-for-HDTF")

In [ ]:
!git clone https://github.com/B-I-T-W-I-S-E-M-I-N-D-S/ditto-train-lmdm-phase-2-v1.git /workspace/ditto-train-lmdm-phase-2-v1
print("Repository cloned to /workspace/ditto-train-lmdm-phase-2-v1")

## 1.2  Verify GPU

In [ ]:
import torch
print("CUDA available :", torch.cuda.is_available())
print("GPU            :", torch.cuda.get_device_name(0))
print("CUDA version   :", torch.version.cuda)
!python --version

## 1.3  Install Miniconda

In [ ]:
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -q
!bash Miniconda3-latest-Linux-x86_64.sh -b -p /workspace/miniconda
print("Miniconda installed at /workspace/miniconda")

In [ ]:
import os
os.environ["PATH"] = "/workspace/miniconda/bin:" + os.environ["PATH"]

In [ ]:
!/workspace/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/workspace/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!/workspace/miniconda/bin/conda --version

## 1.4  Create Conda Environment (Python 3.10)

In [ ]:
!/workspace/miniconda/bin/conda create -n ditto_train python=3.10 -y
print("Environment ditto_train created")

## 1.5  Install System Dependencies

In [ ]:
# FFmpeg dev headers + portaudio are required to build PyAV (av) from source
!apt-get update -qq && apt-get install -y -qq \
    pkg-config \
    libavformat-dev \
    libavcodec-dev \
    libavdevice-dev \
    libavutil-dev \
    libswscale-dev \
    libswresample-dev \
    libavfilter-dev \
    ffmpeg \
    portaudio19-dev \
    git-lfs
print("System dependencies installed.")

## 1.6  Install Python Packages

In [ ]:
# PyTorch 1.12.1 + CUDA 11.3  (matches the pytorch3d prebuilt wheel below)
!/workspace/miniconda/envs/ditto_train/bin/pip install \
    torch==1.12.1+cu113 \
    torchaudio==0.12.1+cu113 \
    torchvision==0.13.1+cu113 \
    --extra-index-url https://download.pytorch.org/whl/cu113

In [ ]:
# PyTorch3D — prebuilt wheel for py310 / cu113 / pyt1121
!/workspace/miniconda/envs/ditto_train/bin/pip install \
    --extra-index-url https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu113_pyt1121/download.html \
    pytorch3d

In [ ]:
# FFmpeg 4.x via conda-forge (required by av at build time)
!/workspace/miniconda/bin/conda install -y \
    -p /workspace/miniconda/envs/ditto_train \
    -c conda-forge \
    "ffmpeg>=4.4,<5.0" \
    --override-channels -c conda-forge -c defaults

In [ ]:
# Pin Cython 0.29.x so av's .pyx files compile without errors
!/workspace/miniconda/envs/ditto_train/bin/pip install "Cython==0.29.37" --force-reinstall -q

# Build and install PyAV from source
!/workspace/miniconda/envs/ditto_train/bin/pip install \
    "av==10.0.0" \
    --no-build-isolation \
    --no-cache-dir

# Sanity-check
!/workspace/miniconda/envs/ditto_train/bin/python -c "import av; print('av OK:', av.__version__)"

In [ ]:
# Vision, image, and numerical computing
!/workspace/miniconda/envs/ditto_train/bin/pip install \
    numpy==1.26.2 \
    pandas==2.1.1 \
    scipy==1.11.4 \
    "scikit-image==0.22.0" \
    "scikit-learn==1.3.2" \
    matplotlib==3.8.2 \
    "pillow==10.2.0" \
    "imageio==2.33.1" \
    "imageio-ffmpeg==0.4.9" \
    "opencv-python==4.10.0.84" \
    "opencv-contrib-python==4.10.0.84" \
    "opencv-python-headless==4.8.1.78"

In [ ]:
# Audio processing
!/workspace/miniconda/envs/ditto_train/bin/pip install \
    librosa==0.10.1 \
    audioread==3.0.1 \
    soundfile==0.12.1 \
    sounddevice==0.4.6 \
    pyaudio==0.2.11 \
    soxr==0.3.7 \
    resampy==0.4.2 \
    python-speech-features==0.6 \
    praat-parselmouth==0.4.3 \
    numba==0.58.1 \
    llvmlite==0.41.1 \
    moviepy==1.0.3

In [ ]:
# ML / deep learning — HuggingFace, face analysis, ONNX
!/workspace/miniconda/envs/ditto_train/bin/pip install \
    accelerate==0.33.0 \
    transformers==4.36.0 \
    tokenizers==0.15.0 \
    huggingface-hub==0.24.5 \
    safetensors==0.4.1 \
    diffusers \
    einops==0.7.0 \
    timm==0.9.0 \
    kornia==0.7.0 \
    lpips==0.1.4 \
    face-alignment==1.4.1 \
    facenet-pytorch==2.6.0 \
    mediapipe==0.10.8 \
    hsemotion==0.3.0 \
    onnx==1.16.2 \
    onnxruntime-gpu==1.16.3

In [ ]:
# chumpy — setup.py calls pip internally, so we bypass with --no-build-isolation
!/workspace/miniconda/envs/ditto_train/bin/pip install \
    "chumpy==0.70" \
    --no-build-isolation \
    --no-cache-dir

In [ ]:
# 3D utilities, training helpers, and miscellaneous packages
!/workspace/miniconda/envs/ditto_train/bin/pip install \
    trimesh==4.0.5 \
    pymcubes==0.1.4 \
    pykalman==0.9.7 \
    natsort==8.4.0 \
    ninja==1.11.1.1 \
    tensorboard==2.15.1 \
    tensorboardx==2.6.2.2 \
    yacs==0.1.8 \
    configargparse==1.7 \
    dill==0.3.8 \
    multiprocess==0.70.16 \
    tqdm==4.66.1 \
    rich==13.7.0 \
    tyro==0.8.8 \
    tabulate==0.9.0 \
    dearpygui==1.10.1 \
    torch-ema==0.3 \
    scikit-video==1.1.11 \
    fsspec==2023.12.1 \
    filelock==3.13.1 \
    packaging==23.2 \
    psutil==6.0.0

In [ ]:
# Aliyun OSS SDK (cloud storage access)
!/workspace/miniconda/envs/ditto_train/bin/pip install \
    aliyun-python-sdk-core==2.14.0 \
    aliyun-python-sdk-kms==2.16.2 \
    oss2==2.18.3 \
    crcmod==1.7 \
    pycryptodome==3.19.0

## 1.7  Patch chumpy (NumPy Compatibility Fix)

In [ ]:
# chumpy 0.70 uses numpy type aliases removed in NumPy ≥ 1.24.
# Replace them with their modern equivalents.
env_path = "/workspace/miniconda/envs/ditto_train"
chumpy_init = f"{env_path}/lib/python3.10/site-packages/chumpy/__init__.py"

with open(chumpy_init, "r") as f:
    content = f.read()

old_line = "from numpy import bool, int, float, complex, object, unicode, str, nan, inf"
new_line = (
    "from numpy import nan, inf\n"
    "import numpy as _np\n"
    "bool=_np.bool_; int=_np.int_; float=_np.float64; "
    "complex=_np.complex128; object=_np.object_; unicode=_np.str_; str=_np.str_"
)

if old_line in content:
    content = content.replace(old_line, new_line)
    with open(chumpy_init, "w") as f:
        f.write(content)
    print("chumpy patched successfully")
else:
    print("chumpy already patched or line not found — skipping")

## 1.8  Register Jupyter Kernel

> ⚠️ **After running this cell:**  
> 1. Refresh the browser page  
> 2. Switch the kernel to **"Python (ditto_train)"**  
> 3. Then continue with Section 2

In [ ]:
!/workspace/miniconda/envs/ditto_train/bin/pip install ipykernel -q
!/workspace/miniconda/envs/ditto_train/bin/python -m ipykernel install \
    --user --name ditto_train --display-name "Python (ditto_train)"
print("")
print("========================================================")
print(" Kernel registered!")
print(" → Refresh this page")
print(" → Select kernel: Python (ditto_train)")
print(" → Then run Section 2 onwards")
print("========================================================")

## 1.9  Verify Environment

> Run after switching to the **ditto_train** kernel.

In [ ]:
import subprocess
result = subprocess.run(
    [
        "/workspace/miniconda/envs/ditto_train/bin/python", "-c",
        "import torch; print('torch    :', torch.__version__); "
        "print('CUDA     :', torch.cuda.is_available()); "
        "import av; print('av       :', av.__version__); "
        "import librosa; print('librosa  :', librosa.__version__); "
        "import mediapipe; print('mediapipe:', mediapipe.__version__); "
        "import onnxruntime; print('onnxrt   :', onnxruntime.__version__)",
    ],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[:800])

---
# 2  Checkpoints Preparation

Download the pretrained Ditto model weights from HuggingFace.  
Only the `ditto_pytorch` sub-directory is required for data preprocessing and training.

```
checkpoints/
└── ditto_pytorch/
    ├── aux_models/
    │   ├── 2d106det.onnx
    │   ├── det_10g.onnx
    │   ├── face_landmarker.task
    │   ├── hubert_streaming_fix_kv.onnx
    │   └── landmark203.onnx
    └── models/
        ├── appearance_extractor.pth
        ├── decoder.pth
        ├── motion_extractor.pth
        ├── stitch_network.pth
        └── warp_network.pth
```

In [ ]:
# Initialise Git LFS (required to download large model files)
!git lfs install
print("Git LFS initialised")


In [ ]:
# Clone the full HuggingFace model repo into the checkpoints/ folder.
# This may take several minutes depending on network speed (~several GB).
!git clone https://huggingface.co/digital-avatar/ditto-talkinghead \
    /workspace/ditto-train-lmdm-phase-2-v1/checkpoints
print("Checkpoints downloaded")

In [ ]:
# Verify that the required checkpoint files are present
import os

CKPT_BASE = "/workspace/ditto-train-lmdm-phase-2-v1/checkpoints/ditto_pytorch"

required_files = [
    "aux_models/2d106det.onnx",
    "aux_models/det_10g.onnx",
    "aux_models/face_landmarker.task",
    "aux_models/hubert_streaming_fix_kv.onnx",
    "aux_models/landmark203.onnx",
    "models/appearance_extractor.pth",
    "models/decoder.pth",
    "models/motion_extractor.pth",
    "models/stitch_network.pth",
    "models/warp_network.pth",
]

all_ok = True
for f in required_files:
    full = os.path.join(CKPT_BASE, f)
    status = "✅" if os.path.isfile(full) else "❌ MISSING"
    print(f"  {status}  {f}")
    if "MISSING" in status:
        all_ok = False

print()
print("All checkpoints present" if all_ok else "⚠️  Some checkpoints are missing — check the download above")

---
# 3  Data Preparation

Converts raw MP4 videos → all feature files needed for training.

**Pipeline:**
```
Raw videos (25 fps MP4 with audio)
    │
    ├─ [Step 1]  Crop video via LivePortrait face detector  → video_list
    ├─ [Step 2]  Extract audio                              → wav_list
    ├─ [Step 3]  HuBERT audio features                      → hubert_aud_npy_list
    ├─ [Step 4]  LP motion features (normal + flipped)      → LP_pkl_list, LP_npy_list
    ├─ [Step 5]  MediaPipe eye features (normal + flipped)  → MP_lmk_npy_list, eye_open/ball
    ├─ [Step 6]  Emotion features                           → emo_npy_list
    ├─ [Step 7]  Gather → data_list.json
    └─ [Step 8]  Preload cache → data_preload.pkl
```

> **Before using your own data:** Ensure all videos are pre-cleaned, shot-detected,  
> normalised to **25 fps MP4** with synchronised audio, and show a clear target face.

## 3.1  Configure Paths

In [ ]:
import os

# ── Core paths ────────────────────────────────────────────────────────────────
DITTO_PATH     = "/workspace/ditto-train-lmdm-phase-2-v1"
PYTHON         = "/workspace/miniconda/envs/ditto_train/bin/python"

# ── Checkpoint paths ──────────────────────────────────────────────────────────
DITTO_PYTORCH_PATH = f"{DITTO_PATH}/checkpoints/ditto_pytorch"
HUBERT_ONNX        = f"{DITTO_PYTORCH_PATH}/aux_models/hubert_streaming_fix_kv.onnx"
MP_FACE_LMK_TASK   = f"{DITTO_PYTORCH_PATH}/aux_models/face_landmarker.task"

# ── Data paths — change these if using your own dataset ───────────────────────
# To use the built-in example dataset, leave as-is.
# To use your own data, point DATA_DIR at your video folder and
# update / recreate data_info.json accordingly (see Section 3.2).
DATA_DIR         = f"{DITTO_PATH}/example/trainset_example"
DATA_INFO_JSON   = f"{DATA_DIR}/data_info.json"
DATA_LIST_JSON   = f"{DATA_DIR}/data_list.json"
DATA_PRELOAD_PKL = f"{DATA_DIR}/data_preload.pkl"

# Export for shell cells
os.environ["DITTO_PATH"]          = DITTO_PATH
os.environ["PYTHON"]              = PYTHON
os.environ["DITTO_PYTORCH_PATH"]  = DITTO_PYTORCH_PATH
os.environ["HUBERT_ONNX"]         = HUBERT_ONNX
os.environ["MP_FACE_LMK_TASK"]    = MP_FACE_LMK_TASK
os.environ["DATA_INFO_JSON"]      = DATA_INFO_JSON
os.environ["DATA_LIST_JSON"]      = DATA_LIST_JSON
os.environ["DATA_PRELOAD_PKL"]    = DATA_PRELOAD_PKL

print("Paths configured:")
for k, v in {
    "DITTO_PATH":        DITTO_PATH,
    "DITTO_PYTORCH_PATH": DITTO_PYTORCH_PATH,
    "DATA_DIR":          DATA_DIR,
    "DATA_INFO_JSON":    DATA_INFO_JSON,
    "DATA_LIST_JSON":    DATA_LIST_JSON,
    "DATA_PRELOAD_PKL":  DATA_PRELOAD_PKL,
}.items():
    print(f"  {k:<22} = {v}")

## 3.2  Build `data_info.json`

For the **example dataset** the script auto-generates this file.  
For **your own dataset**, edit the script or create `data_info.json` manually  
following the structure in the README (all paths must be absolute).

In [ ]:
!$PYTHON $DITTO_PATH/example/get_data_info_json_for_trainset_example.py
print(f"data_info.json saved to: {DATA_INFO_JSON}")

## 3.3  Run Feature Extraction Pipeline

This runs all 8 steps of `prepare_data.sh` using the updated Linux-compatible script.

In [ ]:
# Step 0: Verify checkpoints
!$PYTHON $DITTO_PATH/prepare_data/scripts/check_ckpt_path.py \
    --ditto_pytorch_path $DITTO_PYTORCH_PATH

In [ ]:
# !which ffmpeg || echo "ffmpeg not in PATH"
# !$PYTHON -c "import imageio_ffmpeg; print(imageio_ffmpeg.get_ffmpeg_exe())"

In [ ]:
# !apt-get install -y ffmpeg

In [ ]:
# import os
# os.environ["IMAGEIO_FFMPEG_EXE"] = "/usr/bin/ffmpeg"

In [ ]:
!$PYTHON -m pip install --force-reinstall setuptools

In [ ]:
!$PYTHON -m pip install --force-reinstall imageio-ffmpeg

In [ ]:
!$PYTHON -c "import imageio_ffmpeg; print(imageio_ffmpeg.get_ffmpeg_exe())"

In [ ]:
# Step 1: Crop video via LivePortrait face detector  (fps25_video_list → video_list)
!$PYTHON $DITTO_PATH/prepare_data/scripts/crop_video_by_LP.py \
    -i $DATA_INFO_JSON \
    --ditto_pytorch_path $DITTO_PYTORCH_PATH

In [ ]:
# Step 2: Extract audio from video  (video_list → wav_list)
!$PYTHON $DITTO_PATH/prepare_data/scripts/extract_audio_from_video.py \
    -i $DATA_INFO_JSON

In [ ]:
!$PYTHON -m pip install --force-reinstall setuptools

In [ ]:
!$PYTHON -c "import pkg_resources; print('pkg_resources OK:', pkg_resources.__version__)"

In [ ]:
!$PYTHON -m pip install --force-reinstall setuptools==69.5.1

In [ ]:
# Step 3: HuBERT audio features  (wav_list → hubert_aud_npy_list)
!$PYTHON $DITTO_PATH/prepare_data/scripts/extract_audio_feat_by_Hubert.py \
    -i $DATA_INFO_JSON \
    --Hubert_onnx $HUBERT_ONNX

In [ ]:
# Step 4a: LP motion features — normal  (video_list → LP_pkl_list, LP_npy_list)
!$PYTHON $DITTO_PATH/prepare_data/scripts/extract_motion_feat_by_LP.py \
    -i $DATA_INFO_JSON \
    --ditto_pytorch_path $DITTO_PYTORCH_PATH

In [ ]:
# Step 4b: LP motion features — flipped
!$PYTHON $DITTO_PATH/prepare_data/scripts/extract_motion_feat_by_LP.py \
    -i $DATA_INFO_JSON \
    --ditto_pytorch_path $DITTO_PYTORCH_PATH \
    --flip_flag

In [ ]:
# Step 5a: Eye ratio features — normal  (video_list → MP_lmk, eye_open, eye_ball)
!$PYTHON $DITTO_PATH/prepare_data/scripts/extract_eye_ratio_from_video.py \
    -i $DATA_INFO_JSON \
    --MP_face_landmarker_task_path $MP_FACE_LMK_TASK

In [ ]:
# Step 5b: Eye ratio features — flipped
!$PYTHON $DITTO_PATH/prepare_data/scripts/extract_eye_ratio_from_video.py \
    -i $DATA_INFO_JSON \
    --MP_face_landmarker_task_path $MP_FACE_LMK_TASK \
    --flip_lmk_flag

In [ ]:
# Step 6: Emotion features  (video_list → emo_npy_list)
!$PYTHON $DITTO_PATH/prepare_data/scripts/extract_emo_feat_from_video.py \
    -i $DATA_INFO_JSON

In [ ]:
# Step 7: Gather data_list.json for training
!$PYTHON $DITTO_PATH/prepare_data/scripts/gather_data_list_json_for_train.py \
    -i $DATA_INFO_JSON \
    -o $DATA_LIST_JSON \
    --use_emo \
    --use_eye_open \
    --use_eye_ball \
    --with_flip
print(f"data_list.json saved to: {DATA_LIST_JSON}")

In [ ]:
# Step 8: Preload training data into .pkl cache (recommended for faster training)
!$PYTHON $DITTO_PATH/prepare_data/scripts/preload_train_data_to_pkl.py \
    --data_list_json  $DATA_LIST_JSON \
    --data_preload_pkl $DATA_PRELOAD_PKL \
    --use_sc \
    --use_emo \
    --use_eye_open \
    --use_eye_ball \
    --motion_feat_dim 265
print(f"data_preload.pkl saved to: {DATA_PRELOAD_PKL}")

In [ ]:
# Verify output files exist
import os, json

for label, path in [
    ("data_info.json",   DATA_INFO_JSON),
    ("data_list.json",   DATA_LIST_JSON),
    ("data_preload.pkl", DATA_PRELOAD_PKL),
]:
    exists = os.path.isfile(path)
    size   = f"{os.path.getsize(path) / 1024:.1f} KB" if exists else "—"
    status = "✅" if exists else "❌ MISSING"
    print(f"  {status}  {label:<20}  {size}")

---
# 4  Model Training

Trains the **MotionDiT** diffusion model using HuggingFace Accelerate.

**Two presets are provided:**

| Preset | Batch | Epochs | Purpose |
|--------|-------|--------|---------|
| Quick-start (example data) | 100 | 3 | End-to-end pipeline smoke test |
| Full training (your data)  | 1024 | 500 | Production-quality model |

> Training outputs are saved to `${EXP_DIR}/${EXP_NAME}/`

## 4.1  Configure Accelerate

In [ ]:
# Write a non-interactive Accelerate config for single-GPU training on RunPod.
# Edit num_processes if you have multiple GPUs.
import os

accel_config_dir = os.path.expanduser("~/.cache/huggingface/accelerate")
os.makedirs(accel_config_dir, exist_ok=True)

accel_config = """compute_environment: LOCAL_MACHINE
debug: false
distributed_type: 'NO'        # changed from 'NO'
downcast_bf16: 'no'
enable_cpu_affinity: false
gpu_ids: all
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1                      # change to your GPU count (2, 4, 8, etc.)
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false
"""

config_path = os.path.join(accel_config_dir, "default_config.yaml")
with open(config_path, "w") as f:
    f.write(accel_config)

print(f"Accelerate config written to: {config_path}")
print("  distributed_type : NO  (single GPU)")
print("  mixed_precision  : fp16")
print("  num_processes    : 1")
print("")
print("Edit num_processes above for multi-GPU training.")

## 4.2  Configure Training Parameters

In [ ]:
import os

# ── Paths (inherit from Section 3 — re-run Section 3.1 if kernel was restarted)
DITTO_PATH       = "/workspace/ditto-train-lmdm-phase-2-v1"
PYTHON           = "/workspace/miniconda/envs/ditto_train/bin/python"
ACCELERATE       = "/workspace/miniconda/envs/ditto_train/bin/accelerate"
DATA_LIST_JSON   = f"{DITTO_PATH}/example/trainset_example/data_list.json"
DATA_PRELOAD_PKL = f"{DITTO_PATH}/example/trainset_example/data_preload.pkl"

# ── Experiment output ─────────────────────────────────────────────────────────
EXP_DIR  = f"{DITTO_PATH}/example/exp_dir"
EXP_NAME = "exp_trainset_example"          # change this for each new run

# ── Training hyperparameters ──────────────────────────────────────────────────
# Quick-start (example data): batch_size=100, epochs=3
# Full training (your data) : batch_size=1024, epochs=500
BATCH_SIZE  = 1024
NUM_WORKERS = 16
EPOCHS      = 150
SAVE_FREQ   = 5      # save checkpoint every N epochs

# ── Fixed model architecture flags (do not change) ───────────────────────────
AUDIO_FEAT_DIM  = 1103
MOTION_FEAT_DIM = 265

# Export for shell cells
for k, v in {
    "DITTO_PATH": DITTO_PATH, "ACCELERATE": ACCELERATE,
    "DATA_LIST_JSON": DATA_LIST_JSON, "DATA_PRELOAD_PKL": DATA_PRELOAD_PKL,
    "EXP_DIR": EXP_DIR, "EXP_NAME": EXP_NAME,
    "BATCH_SIZE": str(BATCH_SIZE), "NUM_WORKERS": str(NUM_WORKERS),
    "EPOCHS": str(EPOCHS), "SAVE_FREQ": str(SAVE_FREQ),
    "AUDIO_FEAT_DIM": str(AUDIO_FEAT_DIM), "MOTION_FEAT_DIM": str(MOTION_FEAT_DIM),
}.items():
    os.environ[k] = v

os.makedirs(EXP_DIR, exist_ok=True)

print("Training configuration:")
print(f"  EXP_DIR          : {EXP_DIR}")
print(f"  EXP_NAME         : {EXP_NAME}")
print(f"  DATA_LIST_JSON   : {DATA_LIST_JSON}")
print(f"  DATA_PRELOAD_PKL : {DATA_PRELOAD_PKL}")
print(f"  batch_size       : {BATCH_SIZE}")
print(f"  epochs           : {EPOCHS}")
print(f"  audio_feat_dim   : {AUDIO_FEAT_DIM}")
print(f"  motion_feat_dim  : {MOTION_FEAT_DIM}")

In [ ]:
import os

# ── Fine-Tuning Hyperparameters (New) ────────────────────────────────────────

# The checkpoint of your pre-trained base model to fine-tune from
CHECKPOINT_PATH  = f"/workspace/ditto-train-lmdm-phase-2-v1/checkpoints/ditto_pytorch/models/lmdm_v0.4_hubert.pth" # Update this!

# Fine-tuning learning rate and gradient clipping (to prevent explosion)
FINETUNE_LR      = 5e-5
GRAD_CLIP        = 1.0

# Mouth neutralization and loss weighting
# Replace this string with the actual space-separated indices of your mouth dimensions
MOUTH_DIMS       = "20 21 22 23 24 25" # <--- IMPORTANT: update with your exact indices
MOUTH_WEIGHT     = 2.0

# Sync proxy loss weight
SYNC_PROXY_WEIGHT= 0.1

# Export for shell cells
for k, v in {
    "CHECKPOINT_PATH": CHECKPOINT_PATH,
    "FINETUNE_LR": str(FINETUNE_LR),
    "GRAD_CLIP": str(GRAD_CLIP),
    "MOUTH_DIMS": MOUTH_DIMS,
    "MOUTH_WEIGHT": str(MOUTH_WEIGHT),
    "SYNC_PROXY_WEIGHT": str(SYNC_PROXY_WEIGHT),
}.items():
    os.environ[k] = v

print("Fine-Tuning Configuration:")
print(f"  CHECKPOINT_PATH : {CHECKPOINT_PATH}")
print(f"  FINETUNE_LR     : {FINETUNE_LR}")
print(f"  GRAD_CLIP       : {GRAD_CLIP}")
print(f"  MOUTH_DIMS      : {MOUTH_DIMS}")
print(f"  MOUTH_WEIGHT    : {MOUTH_WEIGHT}")

## 4.3  Launch Training

In [ ]:
# %%bash
# cd "${DITTO_PATH}/MotionDiT"

# $ACCELERATE launch train.py \
#     --experiment_dir     "${EXP_DIR}" \
#     --experiment_name    "${EXP_NAME}" \
#     --use_sc \
#     --use_last_frame \
#     --use_last_frame_loss \
#     --use_emo \
#     --use_eye_open \
#     --use_eye_ball \
#     --audio_feat_dim     "${AUDIO_FEAT_DIM}" \
#     --motion_feat_dim    "${MOTION_FEAT_DIM}" \
#     --batch_size         "${BATCH_SIZE}" \
#     --num_workers        "${NUM_WORKERS}" \
#     --epochs             "${EPOCHS}" \
#     --save_ckpt_freq     "${SAVE_FREQ}" \
#     --data_list_json     "${DATA_LIST_JSON}" \
#     --data_preload \
#     --data_preload_pkl   "${DATA_PRELOAD_PKL}"

In [ ]:
%%bash
cd "${DITTO_PATH}/MotionDiT"

# Create a new experiment directory name for the fine-tuning run
FINETUNE_EXP_NAME="${EXP_NAME}_finetune"

$ACCELERATE launch train.py \
    --experiment_dir          "${EXP_DIR}" \
    --experiment_name         "${FINETUNE_EXP_NAME}" \
    --use_sc \
    --use_last_frame \
    --use_last_frame_loss \
    --use_emo \
    --use_eye_open \
    --use_eye_ball \
    --audio_feat_dim          "${AUDIO_FEAT_DIM}" \
    --motion_feat_dim         "${MOTION_FEAT_DIM}" \
    --batch_size              "${BATCH_SIZE}" \
    --num_workers             "${NUM_WORKERS}" \
    --epochs                  "${EPOCHS}" \
    --save_ckpt_freq          "${SAVE_FREQ}" \
    --data_list_json          "${DATA_LIST_JSON}" \
    --data_preload \
    --data_preload_pkl        "${DATA_PRELOAD_PKL}" \
    --checkpoint              "${CHECKPOINT_PATH}" \
    --no-strict_checkpoint \
    --finetune_mode \
    --finetune_lr             "${FINETUNE_LR}" \
    --grad_clip_max_norm      "${GRAD_CLIP}" \
    --neutralize_mouth_ref \
    --mouth_dims              ${MOUTH_DIMS} \
    --mouth_group_weight      "${MOUTH_WEIGHT}" \
    --use_sync_proxy_loss \
    --lambda_sync_proxy       "${SYNC_PROXY_WEIGHT}"


 95%|█████████▌| 125/131 [01:03<00:02,  2.16it/s]


In [ ]:
!ls /workspace/ditto-train-lmdm-phase-2-v1/checkpoints/ditto_pytorch/models/lmdm_v0.4_hubert.pth

In [ ]:
# %%bash
# cd "${DITTO_PATH}/MotionDiT"

# # Create a new experiment directory name for the fine-tuning run
# FINETUNE_EXP_NAME="${EXP_NAME}_finetune"

# $ACCELERATE launch train.py \
#     --experiment_dir          "${EXP_DIR}" \
#     --experiment_name         "${FINETUNE_EXP_NAME}" \
#     --use_sc \
#     --use_last_frame \
#     --use_last_frame_loss \
#     --use_emo \
#     --use_eye_open \
#     --use_eye_ball \
#     --audio_feat_dim          "${AUDIO_FEAT_DIM}" \
#     --motion_feat_dim         "${MOTION_FEAT_DIM}" \
#     --batch_size              "${BATCH_SIZE}" \
#     --num_workers             "${NUM_WORKERS}" \
#     --epochs                  "${EPOCHS}" \
#     --save_ckpt_freq          "${SAVE_FREQ}" \
#     --data_list_json          "${DATA_LIST_JSON}" \
#     --data_preload \
#     --data_preload_pkl        "${DATA_PRELOAD_PKL}" \
#     --checkpoint              "${CHECKPOINT_PATH}" \
#     --strict_checkpoint       False \
#     --finetune_mode \
#     --finetune_lr             "${FINETUNE_LR}" \
#     --grad_clip_max_norm      "${GRAD_CLIP}" \
#     --neutralize_mouth_ref \
#     --mouth_dims              ${MOUTH_DIMS} \
#     --mouth_group_weight      "${MOUTH_WEIGHT}" \
#     --use_sync_proxy_loss \
#     --lambda_sync_proxy       "${SYNC_PROXY_WEIGHT}"


In [ ]:
# !rm -r /workspace/ditto-train-lmdm-lip-only-v1/example/exp_dir

## 4.4  Monitor Training (TensorBoard)

In [ ]:
import os
EXP_DIR  = os.environ.get("EXP_DIR",  "/workspace/ditto-train-lmdm-lip-only-v1/example/exp_dir")
EXP_NAME = os.environ.get("EXP_NAME", "exp_trainset_example")
log_dir  = os.path.join(EXP_DIR, EXP_NAME)

%load_ext tensorboard
%tensorboard --logdir {log_dir} --port 6006

## 4.5  Check Output Checkpoints

In [ ]:
import os, glob

EXP_DIR  = os.environ.get("EXP_DIR",  "/workspace/ditto-train-fixed-for-HDTF/example/exp_dir")
EXP_NAME = os.environ.get("EXP_NAME", "exp_trainset_example")
out_dir  = os.path.join(EXP_DIR, EXP_NAME)

ckpts = sorted(glob.glob(os.path.join(out_dir, "**", "*.pth"), recursive=True))

if ckpts:
    print(f"Checkpoints found in {out_dir}:")
    for c in ckpts:
        size = os.path.getsize(c) / (1024**2)
        print(f"  {os.path.relpath(c, out_dir):<50}  {size:.1f} MB")
else:
    print(f"No .pth files found under {out_dir}")
    print("Training may still be running, or the experiment name is different.")